# Chapter 18 – Recursive versus iterative (Java)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative_java.ipynb)

The algorithms of the chapter, recursive and iterative, compared with **counts** (calls, iterations, comparisons, recursion depth) and with some measured times: **SumExplicitStack** (recursion simulated with a stack), recursive **MergeSort** and **MergeSortBottomUp**, the recursive fast power and **PowerIt**, **Kadane**, the three Fibonacci algorithms **FibRec**, **FibMemo** and **FibIt**, **GCDRec**/**GCDIt** and the recursive and iterative binary search (tail-recursion elimination). The counts are the same in every language and on every machine; the times (in ms) depend on the machine and on the language, only how they grow matters.

The full chapter, with pseudo-code, proofs and worked examples, is [on the website](https://fabiofurini.github.io/algorithms/recursion/18-recursive-vs-iterative/).

Each program is written to a `.java` file by the `%%writefile` cell and then compiled and run with `javac` and `java`: run the cells in order, from top to bottom (the first cell checks or installs the JDK). No external libraries.

**Indices.** In the pseudo-code of the notes arrays are indexed from 1; in the code (as in all the languages used here) from 0: the element `a[i]` of the pseudo-code is the element of index `i-1` in the code, and a range `a[l..r]` becomes `a[l-1..r-1]`. The printed output of the notebooks uses indices from 0.

**Names.** The functions have the name of the pseudo-code, in the convention of the language (as in the introductory chapters): `FibRec` → `fibRec` (camelCase).

In [1]:
# Check (and if needed install) the JDK
!java -version || apt-get -qq install -y default-jdk-headless

openjdk version "25.0.4.1" 2026-08-18
OpenJDK Runtime Environment (build 25.0.4.1+1-1-26.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 25.0.4.1+1-1-26.04.4-Ubuntu, mixed mode, sharing)


## 1. From recursion to an explicit stack: SumExplicitStack

**SumExplicitStack**$(a,n)$ — *Input*: an array $a[1..n]$ of $n\ge1$ integers. *Output*: the sum $a[1]+\dots+a[n]$. The iterative version of the halving sum: the stack $P$ contains the pairs $(l,r)$ of the sub-arrays still to be summed and does the job of the call stack. The stack never contains more than $\Theta(\log_2 n)$ pairs.

On the array $a=[4,7,1,9]$ of the chapter we print the trace (with indices from 0: the pair $(1,4)$ of the chapter is here $(0,3)$; in each row the top of the stack is on the right) and we compare with `SumIterative`, `sumRecHalving` and `sumRecDecrease` of the code of the course, with the maximum depth of the recursion. Then we print the call stack of `sumRecDecrease(a, 3)` on $a=[4,7,1]$ (example of the chapter): the additions are performed only *after* the base case, while the stack is emptied, and at some point the active calls are $n+1$.

In [2]:
%%writefile ExplicitStack.java
import java.util.*;

public class ExplicitStack {
    static int maxP = 0;   // maximum size of the stack P

    // sumExplicitStack(a, n): sum of a[0..n-1] with an explicit stack
    static long sumExplicitStack(int[] a, int n) {
        long s = 0;
        ArrayDeque<int[]> P = new ArrayDeque<>();   // stack of the sub-arrays to be summed
        P.push(new int[]{0, n - 1});
        maxP = Math.max(maxP, P.size());
        while (!P.isEmpty()) {
            int[] top = P.pop();
            int l = top[0], r = top[1];
            if (l == r) {
                s = s + a[l];           // base case
            } else {
                int mid = l + (r - l) / 2;
                P.push(new int[]{mid + 1, r});
                P.push(new int[]{l, mid});
                maxP = Math.max(maxP, P.size());
            }
        }
        return s;
    }

    // the top of the stack is on the right: we print from the bottom to the top
    static String fmtP(ArrayDeque<int[]> P) {
        StringBuilder sb = new StringBuilder("[");
        Iterator<int[]> it = P.descendingIterator();
        boolean first = true;
        while (it.hasNext()) {
            int[] x = it.next();
            if (!first) sb.append(",");
            sb.append("(").append(x[0]).append(",").append(x[1]).append(")");
            first = false;
        }
        return sb.append("]").toString();
    }

    static long sumExplicitStackTrace(int[] a, int n) {
        long s = 0;
        ArrayDeque<int[]> P = new ArrayDeque<>();
        P.push(new int[]{0, n - 1});
        while (!P.isEmpty()) {
            int[] top = P.pop();
            int l = top[0], r = top[1];
            if (l == r) {
                s = s + a[l];
                System.out.println("pop (" + l + "," + r + "): base case, s=" + s + " => P=" + fmtP(P));
            } else {
                int mid = l + (r - l) / 2;
                P.push(new int[]{mid + 1, r});
                P.push(new int[]{l, mid});
                System.out.println("pop (" + l + "," + r + "): mid=" + mid + ", push (" + (mid + 1) + "," + r + ") and (" + l + "," + mid + ") => P=" + fmtP(P) + ", s=" + s);
            }
        }
        return s;
    }

    // ---- the other sums of the course code, with the depth ----
    static int depth = 0;
    static int maxDepth = 0;

    static long sumIt(int[] a, int n) {
        long s = 0;
        for (int i = 0; i < n; i++) {
            s = s + a[i];
        }
        return s;
    }

    static long sumRecHalving(int[] a, int l, int r) {
        depth++;
        if (depth > maxDepth) maxDepth = depth;
        long result;
        if (l == r) {
            result = a[l];
        } else {
            int mid = l + (r - l) / 2;
            result = sumRecHalving(a, l, mid) + sumRecHalving(a, mid + 1, r);
        }
        depth--;
        return result;
    }

    static long sumRecDecrease(int[] a, int n) {
        depth++;
        if (depth > maxDepth) maxDepth = depth;
        long result;
        if (n == 0) {
            result = 0;
        } else {
            result = sumRecDecrease(a, n - 1) + a[n - 1];
        }
        depth--;
        return result;
    }

    // ---- the call stack of sumRecDecrease(a, 3) ----
    static int stepNo = 0;
    static int maxActive = 0;

    static void emit(ArrayList<String> stack, String action) {
        stepNo++;
        if (stack.size() > maxActive) maxActive = stack.size();
        System.out.printf("%2d | %s | %s%n", stepNo, String.join(" ", stack), action);
    }

    static long sumRecDecreaseTrace(int[] a, int n, ArrayList<String> stack) {
        stack.add("s(" + n + ")");
        if (n == 0) {
            emit(stack, "n=0: return 0");
            stack.remove(stack.size() - 1);
            return 0;
        }
        emit(stack, "n=" + n + " != 0: call s(" + (n - 1) + "), then add a[" + (n - 1) + "]");
        long r = sumRecDecreaseTrace(a, n - 1, stack);
        emit(stack, "return " + r + "+a[" + (n - 1) + "]=" + (r + a[n - 1]));
        stack.remove(stack.size() - 1);
        return r + a[n - 1];
    }

    public static void main(String[] args) {
        int[] a = {4, 7, 1, 9};
        int n = 4;
        System.out.println("a = " + Arrays.toString(a) + "   n = " + n);
        System.out.println("SumExplicitStack(a, 4):");
        long res = sumExplicitStackTrace(a, n);
        maxP = 0;
        sumExplicitStack(a, n);
        System.out.println("result " + res + ", maximum size of P = " + maxP);
        System.out.println();
        System.out.println("sumIt                = " + sumIt(a, n));
        maxDepth = 0;
        long r1 = sumRecHalving(a, 0, n - 1);
        System.out.println("sumRecHalving        = " + r1 + "   maximum depth = " + maxDepth);
        maxDepth = 0;
        long r2 = sumRecDecrease(a, n);
        System.out.println("sumRecDecrease       = " + r2 + "   maximum depth = " + maxDepth);

        int[] b = {4, 7, 1};
        System.out.println();
        System.out.println("a = " + Arrays.toString(b) + "   sumRecDecrease(a, 3)");
        System.out.println(" # | stack of the active calls | action");
        res = sumRecDecreaseTrace(b, 3, new ArrayList<>());
        System.out.println("result " + res + ", active calls at most = " + maxActive + " = n+1");
    }
}

Writing ExplicitStack.java


In [3]:
!javac ExplicitStack.java && java ExplicitStack

a = [4, 7, 1, 9]   n = 4
SumExplicitStack(a, 4):
pop (0,3): mid=1, push (2,3) and (0,1) => P=[(2,3),(0,1)], s=0
pop (0,1): mid=0, push (1,1) and (0,0) => P=[(2,3),(1,1),(0,0)], s=0
pop (0,0): base case, s=4 => P=[(2,3),(1,1)]
pop (1,1): base case, s=11 => P=[(2,3)]
pop (2,3): mid=2, push (3,3) and (2,2) => P=[(3,3),(2,2)], s=11
pop (2,2): base case, s=12 => P=[(3,3)]
pop (3,3): base case, s=21 => P=[]
result 21, maximum size of P = 3

sumIt                = 21
sumRecHalving        = 21   maximum depth = 3
sumRecDecrease       = 21   maximum depth = 5

a = [4, 7, 1]   sumRecDecrease(a, 3)
 # | stack of the active calls | action
 1 | s(3) | n=3 != 0: call s(2), then add a[2]
 2 | s(3) s(2) | n=2 != 0: call s(1), then add a[1]
 3 | s(3) s(2) s(1) | n=1 != 0: call s(0), then add a[0]
 4 | s(3) s(2) s(1) s(0) | n=0: return 0
 5 | s(3) s(2) s(1) | return 0+a[0]=4
 6 | s(3) s(2) | return 4+a[1]=11
 7 | s(3) | return 11+a[2]=12
result 12, active calls at most = 4 = n+1


**Try it yourself.** Change the array: the explicit stack never contains more than $\lceil\log_2 n\rceil+1$ pairs.

In [4]:
%%writefile ProvaExplicitStack.java
import java.util.*;

public class ProvaExplicitStack {
    public static void main(String[] args) {
        // Change the array: the explicit stack never contains more than ceil(log2 n) + 1 pairs.
        int[] c = new int[100];
        for (int i = 0; i < 100; i++) c[i] = i + 1;
        ExplicitStack.maxP = 0;
        long s = ExplicitStack.sumExplicitStack(c, 100);
        System.out.println(s + " " + ExplicitStack.maxP);
    }
}

Writing ProvaExplicitStack.java


In [5]:
!javac ProvaExplicitStack.java && java ProvaExplicitStack

5050 8


## 2. Recursive MergeSort and MergeSortBottomUp

**MergeSortBottomUp**$(a,n)$ — *Input*: an array $a[1..n]$ of natural numbers. *Output*: the array sorted in non-decreasing order. The same splitting as Merge Sort, but **without recursion**: first we merge pairs of sub-arrays of length $1$, then of length $2$, $4$, … with the procedure `Merge` of chapter 17 (repeated here). It executes $\lceil\log_2 n\rceil$ passes and $\Theta(n\log_2 n)$ steps, like the recursive version, and does not use the call stack.

First the example of the chapter $a=[5,2,4,6,1,3]$, pass by pass (indices are from 0). Then the comparison, on pseudo-random arrays (same generator in all languages, seed $12345$, values between $1$ and $10^6$), of the number of **comparisons** between elements of `InsertionSort`, `MergeSort` and `MergeSortBottomUp`: $\Theta(n^2)$ against $\Theta(n\log_2 n)$, with the two versions of Merge Sort practically equal ("OK" means that the three algorithms give the same sorted array), and the measured times.

In [6]:
%%writefile BottomUp.java
import java.util.*;

public class BottomUp {
    static long comparisons = 0;   // comparisons between elements

    // merge(a, l, m, r) of chapter 17
    static void merge(int[] a, int l, int m, int r) {
        int n1 = m - l + 1;
        int n2 = r - m;
        int[] L = new int[n1];
        int[] R = new int[n2];
        for (int i = 0; i < n1; i++) {
            L[i] = a[l + i];
        }
        for (int j = 0; j < n2; j++) {
            R[j] = a[m + 1 + j];
        }
        int i = 0, j = 0, k = l;
        while (i < n1 && j < n2) {
            comparisons++;
            if (L[i] <= R[j]) {
                a[k] = L[i];
                i++;
            } else {
                a[k] = R[j];
                j++;
            }
            k++;
        }
        while (i < n1) {
            a[k] = L[i];
            i++;
            k++;
        }
        while (j < n2) {
            a[k] = R[j];
            j++;
            k++;
        }
    }

    static void mergeSort(int[] a, int l, int r) {
        if (l < r) {
            int m = l + (r - l) / 2;
            mergeSort(a, l, m);
            mergeSort(a, m + 1, r);
            merge(a, l, m, r);
        }
    }

    // mergeSortBottomUp(a, n): sorts a[0..n-1] without recursion
    static void mergeSortBottomUp(int[] a, int n) {
        int width = 1;
        while (width < n) {
            int l = 0;
            while (l + width < n) {                 // pseudo-code: l + width <= n
                int m = l + width - 1;
                int r = Math.min(l + 2 * width - 1, n - 1);
                merge(a, l, m, r);
                l = l + 2 * width;
            }
            width = 2 * width;
        }
    }

    static void mergeSortBottomUpTrace(int[] a, int n) {
        int width = 1;
        int passes = 0;
        while (width < n) {
            int l = 0;
            StringBuilder parts = new StringBuilder();
            while (l + width < n) {
                int m = l + width - 1;
                int r = Math.min(l + 2 * width - 1, n - 1);
                merge(a, l, m, r);
                if (parts.length() > 0) parts.append(", ");
                parts.append("a[" + l + ".." + m + "] with a[" + (m + 1) + ".." + r + "]");
                l = l + 2 * width;
            }
            passes++;
            System.out.println("width=" + width + ": merge " + parts + " => " + Arrays.toString(a));
            width = 2 * width;
        }
        System.out.println("passes = " + passes);
    }

    // insertionSort(a, n) (chapter 3), with the count of comparisons
    static void insertionSort(int[] a, int n) {
        for (int j = 1; j < n; j++) {
            int key = a[j];
            int i = j - 1;
            while (i >= 0) {
                comparisons++;
                if (a[i] > key) {
                    a[i + 1] = a[i];
                    i = i - 1;
                } else {
                    break;
                }
            }
            a[i + 1] = key;
        }
    }

    // Linear congruential generator: same numbers in Python, C++ and Java
    static int[] randomArray(int n, long seed) {
        long x = seed;
        int[] a = new int[n];
        for (int i = 0; i < n; i++) {
            x = (x * 1103515245L + 12345L) % 2147483648L;
            a[i] = 1 + (int) (x % 1000000L);
        }
        return a;
    }

    static double nowMs() {
        return System.nanoTime() / 1e6;
    }

    public static void main(String[] args) {
        int[] a = {5, 2, 4, 6, 1, 3};
        System.out.println("a = " + Arrays.toString(a) + "   n = " + 6);
        mergeSortBottomUpTrace(a, 6);

        System.out.println();
        System.out.printf("%6s %14s %10s %9s  comparisons%n", "n", "InsertionSort", "MergeSort", "BottomUp");
        double[][] times = new double[3][];
        int idx = 0;
        for (int n : new int[]{1000, 2000, 4000}) {
            int[] base = randomArray(n, 12345);
            int[] x1 = base.clone(), x2 = base.clone(), x3 = base.clone();
            comparisons = 0;
            double t0 = nowMs(); insertionSort(x1, n); double t1 = nowMs() - t0;
            long c1 = comparisons;
            comparisons = 0;
            t0 = nowMs(); mergeSort(x2, 0, n - 1); double t2 = nowMs() - t0;
            long c2 = comparisons;
            comparisons = 0;
            t0 = nowMs(); mergeSortBottomUp(x3, n); double t3 = nowMs() - t0;
            long c3 = comparisons;
            int[] ord = base.clone();
            Arrays.sort(ord);
            String ok = (Arrays.equals(x1, x2) && Arrays.equals(x2, x3) && Arrays.equals(x3, ord)) ? "OK" : "ERROR";
            System.out.printf("%6d %14d %10d %9d  %s%n", n, c1, c2, c3, ok);
            times[idx++] = new double[]{n, t1, t2, t3};
        }
        System.out.println();
        System.out.println("Measured times (they depend on the machine):");
        for (double[] t : times) {
            System.out.printf("n=%-6d InsertionSort %10.3f ms   MergeSort %8.3f ms   MergeSortBottomUp %8.3f ms%n", (int) t[0], t[1], t[2], t[3]);
        }
    }
}

Writing BottomUp.java


In [7]:
!javac BottomUp.java && java BottomUp

a = [5, 2, 4, 6, 1, 3]   n = 6
width=1: merge a[0..0] with a[1..1], a[2..2] with a[3..3], a[4..4] with a[5..5] => [2, 5, 4, 6, 1, 3]
width=2: merge a[0..1] with a[2..3] => [2, 4, 5, 6, 1, 3]
width=4: merge a[0..3] with a[4..5] => [1, 2, 3, 4, 5, 6]
passes = 3

     n  InsertionSort  MergeSort  BottomUp  comparisons
  1000         248782       8716      8740  OK
  2000        1010300      19415     19438  OK
  4000        4027558      42874     42876  OK

Measured times (they depend on the machine):
n=1000   InsertionSort      7.056 ms   MergeSort    0.581 ms   MergeSortBottomUp    0.565 ms
n=2000   InsertionSort      2.039 ms   MergeSort    0.211 ms   MergeSortBottomUp    0.284 ms
n=4000   InsertionSort      2.933 ms   MergeSort    0.479 ms   MergeSortBottomUp    0.614 ms


**Try it yourself.** Change $n$ (with $n=8000$ the comparisons of `InsertionSort` quadruple with respect to $n=4000$, those of Merge Sort little more than double).

In [8]:
%%writefile ProvaBottomUp.java
import java.util.*;

public class ProvaBottomUp {
    public static void main(String[] args) {
        // Try n = 8000: the comparisons of InsertionSort quadruple with respect to n = 4000, those of Merge Sort little more than double.
        int n = 8000;
        int[] base = BottomUp.randomArray(n, 12345);
        int[] x1 = base.clone(), x2 = base.clone();
        BottomUp.comparisons = 0;
        BottomUp.insertionSort(x1, n);
        long c1 = BottomUp.comparisons;
        BottomUp.comparisons = 0;
        BottomUp.mergeSortBottomUp(x2, n);
        System.out.println(c1 + " " + BottomUp.comparisons + " " + (Arrays.equals(x1, x2) ? "True" : "False"));
    }
}

Writing ProvaBottomUp.java


In [9]:
!javac ProvaBottomUp.java && java ProvaBottomUp

16115652 93756 True


## 3. Fast power: recursive and PowerIt (square-and-multiply)

- **FastPower**$(x,n)$ (capitolo 14) — recursive: $x^n=(x^{\lfloor n/2\rfloor})^2$ if $n$ is even, $(x^{\lfloor n/2\rfloor})^2\cdot x$ if it is odd. $\Theta(\log_2 n)$ steps, $\lfloor\log_2 n\rfloor+2$ calls (depth $\Theta(\log_2 n)$). The result of the recursive call is stored in $y$ and used twice.
- **PowerIt**$(x,n)$ — *Input*: a number $x$ and a natural number $n$. *Output*: $x^n$. It reads the bits of $n$ from the least significant one: $\lfloor\log_2 n\rfloor+1$ iterations, extra memory $\Theta(1)$; $p\cdot y^k=x^n$ holds at every iteration.

With $x=2$ and $n=13=(1101)_2$ we print the trace of the chapter: $p$ is multiplied by $x$, $x^4$ and $x^8$ (the bits of 13 equal to 1) and $p=x^{13}$ after $4$ iterations. Then the table: same result, $\lfloor\log_2 n\rfloor+1$ iterations against $\lfloor\log_2 n\rfloor+2$ calls, against the $n$ multiplications of the naive loop.

In [10]:
%%writefile Power.java
import java.util.*;

public class Power {
    static int powCalls = 0;   // calls of FastPower
    static int powIters = 0;   // iterations of PowerIt

    // fastPower(x, n): x^n by recursion on the halving of n
    static long fastPower(long x, int n) {
        powCalls++;
        if (n == 0) {
            return 1;                   // base case
        }
        long y = fastPower(x, n / 2);   // one recursive call
        if (n % 2 == 0) {
            return y * y;               // n even
        }
        return x * y * y;               // n odd
    }

    // powerIt(x, n): x^n with square-and-multiply
    static long powerIt(long x, int n) {
        long p = 1;
        long y = x;
        int k = n;
        while (k > 0) {
            powIters++;
            if (k % 2 == 1) {
                p = p * y;
            }
            y = y * y;
            k = k / 2;
        }
        return p;
    }

    static void powerItTrace(long x, int n) {
        System.out.println("PowerIt(" + x + ", " + n + "): " + n + " = (" + Integer.toBinaryString(n) + ")_2");
        long p = 1, y = x;
        int k = n, it = 0;
        while (k > 0) {
            it++;
            String par = k % 2 == 1 ? "odd" : "even";
            if (k % 2 == 1) {
                p = p * y;
            }
            y = y * y;
            System.out.println(it + ". k=" + k + " " + par + ": p=" + p + ", y=" + y + ", k=" + (k / 2));
            k = k / 2;
        }
        System.out.println("iterations = " + it + ", result p = " + p);
    }

    public static void main(String[] args) {
        powerItTrace(2, 13);
        System.out.println();
        System.out.printf("%3s %12s %8s %10s %6s   (iterations, calls, multiplications)%n", "n", "2^n", "PowerIt", "FastPower", "naive");
        int[] ns = {0, 1, 2, 5, 13, 30};
        for (int n : ns) {
            powIters = 0;
            long r1 = powerIt(2, n);
            powCalls = 0;
            long r2 = fastPower(2, n);
            System.out.printf("%3d %12d %8d %10d %6d   %s%n", n, r1, powIters, powCalls, n, (r1 == r2 && r1 == (1L << n)) ? "OK" : "ERROR");
        }
    }
}

Writing Power.java


In [11]:
!javac Power.java && java Power

PowerIt(2, 13): 13 = (1101)_2
1. k=13 odd: p=2, y=4, k=6
2. k=6 even: p=2, y=16, k=3
3. k=3 odd: p=32, y=256, k=1
4. k=1 odd: p=8192, y=65536, k=0
iterations = 4, result p = 8192

  n          2^n  PowerIt  FastPower  naive   (iterations, calls, multiplications)
  0            1        0          1      0   OK
  1            2        1          2      1   OK
  2            4        2          3      2   OK
  5           32        3          4      5   OK
 13         8192        4          5     13   OK
 30   1073741824        5          6     30   OK


**Try it yourself.** Try another base and another exponent (in C++ and Java the result must fit in 64 bits).

In [12]:
%%writefile ProvaPower.java
import java.util.*;

public class ProvaPower {
    public static void main(String[] args) {
        // Try another base and another exponent (beware: in C++ and Java the result must fit in 64 bits).
        Power.powIters = 0;
        Power.powCalls = 0;
        long r1 = Power.powerIt(3, 25);
        long r2 = Power.fastPower(3, 25);
        System.out.println(r1 + " " + Power.powIters + " " + r2 + " " + Power.powCalls);
    }
}

Writing ProvaPower.java


In [13]:
!javac ProvaPower.java && java ProvaPower

847288609443 5 847288609443 6


## 4. Kadane: a better iterative algorithm

**Kadane**$(a,n)$ — *Input*: an array $a[1..n]$ of $n\ge1$ integers. *Output*: the maximum sum of a non-empty subarray $a[l..r]$. It is the compact version of the chapter: $cur\leftarrow\max(a[i],\,cur+a[i])$, $best\leftarrow\max(best,cur)$. The loop makes $n-1$ iterations: $\Theta(n)$ steps and $\Theta(1)$ memory, against the $n(n+1)/2$ pairs $(l,r)$ of brute force ($\Theta(n^2)$) and against the recursive divide and conquer ($\Theta(n\log_2 n)$): a good iterative algorithm can beat recursion.

On the example of the price changes of chapter 17 the result is $43$. Then, on pseudo-random arrays with values between $-50$ and $50$ (same generator in all languages, seed $n$), the iterations of `Kadane`, the pairs examined by brute force and the check that the two algorithms find the same maximum sum; finally the measured times.

In [14]:
%%writefile KadaneCompact.java
import java.util.*;

public class KadaneCompact {
    // kadane(a, n): maximum sum of a non-empty subarray of a[0..n-1]
    static int kadane(int[] a, int n) {
        int best = a[0];
        int cur = a[0];                 // maximum sum of a subarray ending at i
        int i = 1;
        while (i < n) {
            cur = Math.max(a[i], cur + a[i]);
            best = Math.max(best, cur);
            i = i + 1;
        }
        return best;
    }

    // Brute force: returns {maximum sum, pairs (l, r) examined}
    static long[] bruteForceBest(int[] a, int n) {
        int best = a[0];
        long pairs = 0;
        for (int l = 0; l < n; l++) {
            int s = 0;
            for (int r = l; r < n; r++) {
                s = s + a[r];
                pairs++;
                if (s > best) best = s;
            }
        }
        return new long[]{best, pairs};
    }

    // Integers between -50 and 50 (linear congruential generator)
    static int[] randomValues(int n, long seed) {
        long x = seed;
        int[] a = new int[n];
        for (int i = 0; i < n; i++) {
            x = (x * 1103515245L + 12345L) % 2147483648L;
            a[i] = (int) (x % 101L) - 50;
        }
        return a;
    }

    static double nowMs() {
        return System.nanoTime() / 1e6;
    }

    public static void main(String[] args) {
        int[] a = {13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7};
        System.out.println("a = " + Arrays.toString(a));
        System.out.println("Kadane(a, 16) = " + kadane(a, 16));
        int[] b = {-3, -1, -2, -5};
        System.out.println("a = " + Arrays.toString(b) + "  Kadane(a, 4) = " + kadane(b, 4));
        System.out.println();
        System.out.printf("%6s %7s %11s %11s %9s  same sum?%n", "n", "Kadane", "BruteForce", "iterations", "pairs");
        double[][] times = new double[3][];
        int idx = 0;
        for (int n : new int[]{1000, 2000, 4000}) {
            int[] v = randomValues(n, n);
            double t0 = nowMs(); int k = kadane(v, n); double t1 = nowMs() - t0;
            t0 = nowMs(); long[] bf = bruteForceBest(v, n); double t2 = nowMs() - t0;
            System.out.printf("%6d %7d %11d %11d %9d  %s%n", n, k, bf[0], n - 1, bf[1], k == bf[0] ? "OK" : "ERROR");
            times[idx++] = new double[]{n, t1, t2};
        }
        System.out.println();
        System.out.println("Measured times (they depend on the machine):");
        for (double[] t : times) {
            System.out.printf("n=%-6d Kadane %9.4f ms   BruteForce %10.3f ms%n", (int) t[0], t[1], t[2]);
        }
    }
}

Writing KadaneCompact.java


In [15]:
!javac KadaneCompact.java && java KadaneCompact

a = [13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7]
Kadane(a, 16) = 43
a = [-3, -1, -2, -5]  Kadane(a, 4) = -1

     n  Kadane  BruteForce  iterations     pairs  same sum?
  1000     547         547         999    500500  OK
  2000     981         981        1999   2001000  OK
  4000    1955        1955        3999   8002000  OK

Measured times (they depend on the machine):
n=1000   Kadane    0.1613 ms   BruteForce      5.972 ms
n=2000   Kadane    0.1102 ms   BruteForce      0.956 ms
n=4000   Kadane    0.2029 ms   BruteForce      6.742 ms


**Try it yourself.** Change $n$: the pairs of brute force grow as $n^2$, the iterations of Kadane as $n$.

In [16]:
%%writefile ProvaKadaneCompact.java
import java.util.*;

public class ProvaKadaneCompact {
    public static void main(String[] args) {
        // Try n = 8000: the pairs of brute force are n(n+1)/2 = 32 004 000, the iterations of Kadane 7999.
        int n = 8000;
        int[] v = KadaneCompact.randomValues(n, 12345);
        long[] bf = KadaneCompact.bruteForceBest(v, n);
        System.out.println(KadaneCompact.kadane(v, n) + " (" + bf[0] + ", " + bf[1] + ")");
    }
}

Writing ProvaKadaneCompact.java


In [17]:
!javac ProvaKadaneCompact.java && java ProvaKadaneCompact

1903 (1903, 32004000)


## 5. Fibonacci: FibRec, FibMemo and FibIt (overlapping subproblems)

*Input*: a natural number $n$. *Output*: the Fibonacci number $F(n)$, with $F(0)=0$, $F(1)=1$, $F(n)=F(n-1)+F(n-2)$.

- **FibRec**$(n)$ — direct translation of the definition: the subproblems overlap and the calls are $2F(n+1)-1=\Theta(\varphi^n)$, exponential; the depth is $n$.
- **FibMemo**$(n,memo)$ — recursion with memoization: every value is computed once (the vector `memo` is $-1$ for "undefined"); $2n-1$ calls, memory $\Theta(n)$.
- **FibIt**$(n)$ — loop with two variables: $n-1$ iterations, memory $\Theta(1)$.

On the example $\texttt{FibRec}(5)$ of the chapter we count how many times each subproblem is called (the tree has $15$ nodes, but only $6$ different values are needed). Then the table of the counts (which coincide with the values of the chapter: $21\,891$, $242\,785$, $2\,692\,537$ calls for $n=20,25,30$) and the measured times.

In [18]:
%%writefile Fibonacci.java
import java.util.*;

public class Fibonacci {
    static long fibCalls = 0;   // calls executed

    // fibRec(n)
    static long fibRec(int n) {
        fibCalls++;
        if (n <= 1) {
            return n;
        }
        return fibRec(n - 1) + fibRec(n - 2);
    }

    // fibMemo(n, memo): memo[k] = -1 means "undefined"
    static long fibMemo(int n, long[] memo) {
        fibCalls++;
        if (n <= 1) {
            return n;
        }
        if (memo[n] == -1) {
            memo[n] = fibMemo(n - 1, memo) + fibMemo(n - 2, memo);
        }
        return memo[n];
    }

    // fibIt(n)
    static long fibIt(int n) {
        if (n <= 1) {
            return n;
        }
        long prev = 0;                  // F(i-2)
        long curr = 1;                  // F(i-1)
        int i = 2;
        while (i <= n) {
            long next = prev + curr;
            prev = curr;
            curr = next;
            i = i + 1;
        }
        return curr;
    }

    // Like fibRec, but counts the calls for each value of the parameter
    static long fibRecHist(int n, int[] counts) {
        counts[n]++;
        if (n <= 1) {
            return n;
        }
        return fibRecHist(n - 1, counts) + fibRecHist(n - 2, counts);
    }

    static double nowMs() {
        return System.nanoTime() / 1e6;
    }

    public static void main(String[] args) {
        int[] counts = new int[6];
        long f5 = fibRecHist(5, counts);
        System.out.println("FibRec(5) = " + f5);
        StringBuilder line = new StringBuilder();
        int total = 0;
        for (int k = 5; k >= 0; k--) {
            if (k < 5) line.append("  ");
            line.append(k).append(":").append(counts[k]);
            total += counts[k];
        }
        System.out.println("calls for each value of the parameter: " + line + "   total " + total);
        System.out.println();
        System.out.printf("%3s %8s %9s %10s %8s %6s  (calls, calls, calls, iterations)%n", "n", "F(n)", "FibRec", "2F(n+1)-1", "FibMemo", "FibIt");
        int[] ns = {10, 15, 20, 25, 30};
        for (int n : ns) {
            long f = fibIt(n);
            fibCalls = 0;
            long r1 = fibRec(n);
            long c1 = fibCalls;
            fibCalls = 0;
            long[] memo = new long[n + 1];
            Arrays.fill(memo, -1);
            long r2 = fibMemo(n, memo);
            long c2 = fibCalls;
            boolean ok = (r1 == r2 && r2 == f && c1 == 2 * fibIt(n + 1) - 1 && c2 == 2 * n - 1);
            System.out.printf("%3d %8d %9d %10d %8d %6d  %s%n", n, f, c1, 2 * fibIt(n + 1) - 1, c2, n - 1, ok ? "OK" : "ERROR");
        }
        System.out.println();
        System.out.println("Measured times (they depend on the machine):");
        for (int n : new int[]{20, 25, 30}) {
            double t0 = nowMs(); fibRec(n); double t1 = nowMs() - t0;
            t0 = nowMs();
            long sink = 0;
            for (int rep = 0; rep < 1000; rep++) {
                sink += fibIt(n);
            }
            double t2 = (nowMs() - t0) / 1000;
            System.out.printf("n=%-3d FibRec %10.4f ms   FibIt %10.6f ms%n", n, t1, t2);
        }
    }
}

Writing Fibonacci.java


In [19]:
!javac Fibonacci.java && java Fibonacci

FibRec(5) = 5
calls for each value of the parameter: 5:1  4:1  3:2  2:3  1:5  0:3   total 15

  n     F(n)    FibRec  2F(n+1)-1  FibMemo  FibIt  (calls, calls, calls, iterations)
 10       55       177        177       19      9  OK
 15      610      1973       1973       29     14  OK
 20     6765     21891      21891       39     19  OK
 25    75025    242785     242785       49     24  OK
 30   832040   2692537    2692537       59     29  OK

Measured times (they depend on the machine):
n=20  FibRec     0.0335 ms   FibIt   0.000123 ms
n=25  FibRec     0.3514 ms   FibIt   0.000060 ms
n=30  FibRec     3.6161 ms   FibIt   0.000031 ms


**Try it yourself.** Increase $n$ (with $n=32$ `FibRec` makes $7\,049\,155$ calls, `FibMemo` $63$) and see how the time of `FibRec` grows.

In [20]:
%%writefile ProvaFibonacci.java
import java.util.*;

public class ProvaFibonacci {
    public static void main(String[] args) {
        // Increase n: with n = 32 FibRec makes 7 049 155 calls, FibMemo 63.
        int n = 32;
        Fibonacci.fibCalls = 0;
        long r1 = Fibonacci.fibRec(n);
        System.out.println(r1 + " " + Fibonacci.fibCalls);
        Fibonacci.fibCalls = 0;
        long[] memo = new long[n + 1];
        Arrays.fill(memo, -1);
        long r2 = Fibonacci.fibMemo(n, memo);
        System.out.println(r2 + " " + Fibonacci.fibCalls);
    }
}

Writing ProvaFibonacci.java


In [21]:
!javac ProvaFibonacci.java && java ProvaFibonacci

2178309 7049155
2178309 63


## 6. Tail recursion and its elimination: GCDRec/GCDIt, RecBinarySearch/ItBinarySearch

A recursive call is a *tail call* if it is the last operation and its result is returned as it is: it can be replaced by an update of the parameters and a jump to the beginning, that is, by a loop (loop condition = negation of the base case; tail call = simultaneous assignment of the parameters). The result is the same, the number of steps has the same order of growth and the extra memory goes down from $\Theta(\text{depth})$ to $\Theta(1)$.

- **GCDRec**$(a,b)$ / **GCDIt**$(a,b)$ — *Input*: two natural numbers $a$, $b$. *Output*: their greatest common divisor (Euclid's algorithm). With $(a,b)=(252,105)$ both go through the pairs $(252,105)$, $(105,42)$, $(42,21)$, $(21,0)$ and return $21$: four calls against three iterations. The temporary variable $r$ is needed because $a\bmod b$ must be computed with the *old* $a$.
- **RecBS**$(a,l,r,m)$ / **ItBS**$(a,l,r,m)$ — *Input*: a sorted array, two indices $l$, $r$ and a value $m$. *Output*: an index of $m$ in $a[l..r]$ or $-1$. `ItBS(a, 1, n, m)` is the iterative binary search of the chapter "Search".

In [22]:
%%writefile TailRecursion.java
import java.util.*;

public class TailRecursion {
    static ArrayList<String> gcdLog = new ArrayList<>();   // pairs (a, b) visited (only to print them)
    static int rbsCalls = 0;   // calls of RecBS
    static int bsIters = 0;    // iterations of ItBS

    static String pairStr(long a, long b) {
        return "(" + a + "," + b + ")";
    }

    // gcdRec(a, b)
    static long gcdRec(long a, long b) {
        gcdLog.add(pairStr(a, b));
        if (b == 0) {
            return a;
        }
        return gcdRec(b, a % b);            // tail call
    }

    // gcdIt(a, b)
    static long gcdIt(long a, long b) {
        gcdLog.add(pairStr(a, b));
        while (b != 0) {                    // negation of the base case
            long r = a % b;                 // with the old a
            a = b;
            b = r;
            gcdLog.add(pairStr(a, b));
        }
        return a;
    }

    // recBS(a, l, r, m)
    static int recBS(int[] a, int l, int r, int m) {
        rbsCalls++;
        if (l > r) {
            return -1;
        }
        int mid = l + (r - l) / 2;
        if (a[mid] == m) {
            return mid;
        }
        if (a[mid] < m) {
            return recBS(a, mid + 1, r, m);     // tail call
        } else {
            return recBS(a, l, mid - 1, m);     // tail call
        }
    }

    // itBS(a, l, r, m)
    static int itBS(int[] a, int l, int r, int m) {
        while (l <= r) {                        // negation of the base case l > r
            bsIters++;
            int mid = l + (r - l) / 2;
            if (a[mid] == m) {
                return mid;                     // the second base case stays in the loop
            }
            if (a[mid] < m) {
                l = mid + 1;
            } else {
                r = mid - 1;
            }
        }
        return -1;
    }

    public static void main(String[] args) {
        long[][] pairsAB = {{252, 105}, {1071, 462}, {832040, 514229}};
        for (long[] p : pairsAB) {
            long a = p[0], b = p[1];
            gcdLog.clear();
            long g1 = gcdRec(a, b);
            int calls = gcdLog.size();
            String pairs = String.join(" ", gcdLog);
            gcdLog.clear();
            long g2 = gcdIt(a, b);
            int iters = gcdLog.size() - 1;
            System.out.printf("GCDRec(%d, %d) = %d   calls=%d     GCDIt = %d   iterations=%d   %s%n", a, b, g1, calls, g2, iters, g1 == g2 ? "OK" : "ERROR");
            if (a == 252) {
                System.out.println("  pairs: " + pairs);
            }
        }

        System.out.println();
        int[] a = {3, 9, 11, 12, 12, 16, 19};
        int n = 7;
        System.out.println("a = " + Arrays.toString(a) + "   n = " + n);
        System.out.printf("%4s %6s %6s %6s %11s%n", "m", "RecBS", "calls", "ItBS", "iterations");
        int[] ms = {12, 10, 3, 19, 100};
        for (int m : ms) {
            rbsCalls = 0;
            int r1 = recBS(a, 0, n - 1, m);
            bsIters = 0;
            int r2 = itBS(a, 0, n - 1, m);
            System.out.printf("%4d %6d %6d %6d %11d   %s%n", m, r1, rbsCalls, r2, bsIters, r1 == r2 ? "OK" : "ERROR");
        }
    }
}

Writing TailRecursion.java


In [23]:
!javac TailRecursion.java && java TailRecursion

GCDRec(252, 105) = 21   calls=4     GCDIt = 21   iterations=3   OK
  pairs: (252,105) (105,42) (42,21) (21,0)
GCDRec(1071, 462) = 21   calls=4     GCDIt = 21   iterations=3   OK
GCDRec(832040, 514229) = 1   calls=29     GCDIt = 1   iterations=28   OK

a = [3, 9, 11, 12, 12, 16, 19]   n = 7
   m  RecBS  calls   ItBS  iterations
  12      3      1      3           1   OK
  10     -1      4     -1           3   OK
   3      0      3      0           3   OK
  19      6      3      6           3   OK
 100     -1      4     -1           3   OK


**Try it yourself.** Try other pairs for the GCD: with two consecutive Fibonacci numbers Euclid's algorithm makes the largest number of steps.

In [24]:
%%writefile ProvaTailRecursion.java
import java.util.*;

public class ProvaTailRecursion {
    public static void main(String[] args) {
        // Try other pairs for the GCD: with two consecutive Fibonacci numbers Euclid's algorithm makes the largest number of steps.
        TailRecursion.gcdLog.clear();
        long g = TailRecursion.gcdRec(46368, 28657);
        System.out.println(g + " " + TailRecursion.gcdLog.size());
    }
}

Writing ProvaTailRecursion.java


In [25]:
!javac ProvaTailRecursion.java && java ProvaTailRecursion

1 23


## 7. Depth of the recursion and the call stack

The memory used by the call stack is proportional to the **depth** $D(n)$ of the recursion (the maximum number of calls active at the same time), not to the number of calls. The table compares the **measured** depth with the formula of the chapter:

| algorithm | $D(n)$ |
|---|---|
| sum by decrease by one (base case $n=0$) | $n+1$ |
| sum by halving | $\lceil\log_2 n\rceil+1$ |
| FastPower | $\lfloor\log_2 n\rfloor+2$ |
| RecBS (worst case, absent value) | $\le\lfloor\log_2 n\rfloor+2$ |
| FibRec$(n)$, $n\ge1$ | $n$ |

A depth $\Theta(\log_2 n)$ is never a problem in practice; a depth $\Theta(n)$ risks a **stack overflow**: the last part of the cell (the lines starting with `~~`) depends on the language and on the machine.

In [26]:
%%writefile Depth.java
import java.util.*;

public class Depth {
    static int depth = 0;
    static int maxDepth = 0;
    static int calls = 0;

    static void enter() {
        depth++;
        if (depth > maxDepth) maxDepth = depth;
    }

    static long sumRecDecrease(int[] a, int n) {
        enter();
        long result = (n == 0) ? 0 : sumRecDecrease(a, n - 1) + a[n - 1];
        depth--;
        return result;
    }

    static long sumRecHalving(int[] a, int l, int r) {
        enter();
        long result;
        if (l == r) {
            result = a[l];
        } else {
            int mid = l + (r - l) / 2;
            result = sumRecHalving(a, l, mid) + sumRecHalving(a, mid + 1, r);
        }
        depth--;
        return result;
    }

    static long fastPower(long x, int n) {
        enter();
        long result;
        if (n == 0) {
            result = 1;
        } else {
            long y = fastPower(x, n / 2);
            result = (n % 2 == 0) ? y * y : x * y * y;
        }
        depth--;
        return result;
    }

    static int recBS(int[] a, int l, int r, int m) {
        calls++;
        if (l > r) {
            return -1;
        }
        int mid = l + (r - l) / 2;
        if (a[mid] == m) {
            return mid;
        }
        if (a[mid] < m) {
            return recBS(a, mid + 1, r, m);
        }
        return recBS(a, l, mid - 1, m);
    }

    static long fibRec(int n) {
        enter();
        long result = (n <= 1) ? n : fibRec(n - 1) + fibRec(n - 2);
        depth--;
        return result;
    }

    static int floorLog2(int n) {
        int f = 0;
        while ((1 << (f + 1)) <= n) f++;
        return f;
    }

    static int ceilLog2(int n) {
        int c = 0;
        while ((1 << c) < n) c++;
        return c;
    }

    public static void main(String[] args) {
        System.out.printf("%5s %9s %5s %8s %8s %10s %8s %6s %6s%n", "n", "decrease", "n+1", "halving", "formula", "FastPower", "formula", "RecBS", "bound");
        int[] ns = {10, 100, 500};
        for (int n : ns) {
            int[] a = new int[n];
            for (int i = 0; i < n; i++) a[i] = i;
            maxDepth = 0; sumRecDecrease(a, n); int d1 = maxDepth;
            maxDepth = 0; sumRecHalving(a, 0, n - 1); int d2 = maxDepth;
            maxDepth = 0; fastPower(1, n); int d3 = maxDepth;
            calls = 0; recBS(a, 0, n - 1, n + 100); int d4 = calls;
            System.out.printf("%5d %9d %5d %8d %8d %10d %8d %6d %6d%n", n, d1, n + 1, d2, ceilLog2(n) + 1, d3, floorLog2(n) + 2, d4, floorLog2(n) + 2);
        }
        System.out.println();
        System.out.printf("%3s %13s%n", "n", "FibRec depth");
        int[] fs = {5, 10, 20};
        for (int n : fs) {
            maxDepth = 0;
            fibRec(n);
            System.out.printf("%3d %13d%n", n, maxDepth);
        }

        try {
            depth = 0;
            sumRecDecrease(new int[1000000], 1000000);
            System.out.println("~~ Java: n = 1000000 OK");
        } catch (StackOverflowError e) {
            depth = 0;
            System.out.println("~~ Java: StackOverflowError for sumRecDecrease with n = 1000000 (depth n+1)");
        }
    }
}

Writing Depth.java


In [27]:
!javac Depth.java && java Depth

    n  decrease   n+1  halving  formula  FastPower  formula  RecBS  bound
   10        11    11        5        5          5        5      5      5
  100       101   101        8        8          8        8      8      8
  500       501   501       10       10         10       10     10     10

  n  FibRec depth
  5             5
 10            10
 20            20
~~ Java: StackOverflowError for sumRecDecrease with n = 1000000 (depth n+1)


**Try it yourself.** Measure the depth for another $n$ (here $2000$): `decrease` $=n+1$, `halving` $=\lceil\log_2 n\rceil+1$.

In [28]:
%%writefile ProvaDepth.java
import java.util.*;

public class ProvaDepth {
    public static void main(String[] args) {
        // Measure the depth of the recursion for another n (here 2000): decrease = n+1, halving = ceil(log2 n) + 1.
        int n = 2000;
        int[] a = new int[n];
        Arrays.fill(a, 1);
        Depth.maxDepth = 0;
        Depth.sumRecDecrease(a, n);
        System.out.print(Depth.maxDepth + " ");
        Depth.maxDepth = 0;
        Depth.sumRecHalving(a, 0, n - 1);
        System.out.println(Depth.maxDepth);
    }
}

Writing ProvaDepth.java


In [29]:
!javac ProvaDepth.java && java ProvaDepth

2001 12
